# 03 — FinLLaMA + LLaMA-3.1 Inference (T4, 4-bit)
Runs the full matrix: {FinLLaMA, LLaMA-3.1} × {FPB, FiQA} × {A, B} × {0, 3, 5}.

Checkpoints every 100 samples. Safe to interrupt and re-run — resumes from last checkpoint.

In [ ]:
import sys, os, json, time, hashlib
# Colab: uncomment
# from google.colab import drive, userdata
# drive.mount('/content/drive')
# PROJECT_DIR = '/content/drive/MyDrive/finllama-sentiment'
# os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
# os.environ['TRANSFORMERS_CACHE'] = '/content/drive/MyDrive/hf_cache'
# sys.path.insert(0, PROJECT_DIR); os.chdir(PROJECT_DIR)
# from huggingface_hub import login; login(userdata.get('HF_TOKEN'))

import torch
assert torch.cuda.is_available(), 'Switch to T4 GPU runtime!'

from src.data_loader import load_fpb, load_fiqa
from src.prompts import build_prompt, sample_fewshot
from src.parser import parse
from src.evaluation import compute_metrics
from src.models.llm_runner import LLMRunner
from src.utils import load_config, set_seed

cfg = load_config()
set_seed(cfg['seed'])
PRED_DIR = cfg['paths']['predictions_dir']
os.makedirs(PRED_DIR, exist_ok=True)

In [ ]:
fpb_train, fpb_test = load_fpb(
    config=cfg['datasets']['fpb']['config'],
    test_fraction=cfg['datasets']['fpb']['test_fraction'],
    seed=cfg['seed'],
)
fiqa_test = load_fiqa(neutral_band=cfg['datasets']['fiqa']['neutral_band'])
datasets = {'FPB': (fpb_test, fpb_train), 'FiQA': (fiqa_test, fpb_train)}

In [ ]:
CHECKPOINT_EVERY = 100

def _run_id(model_key, ds_name, template, shots):
    return f'{model_key}__{ds_name}__{template}__{shots}shot__seed{cfg["seed"]}'

def _load_progress(run_dir):
    p = os.path.join(run_dir, 'progress.json')
    if os.path.exists(p):
        with open(p) as f: return json.load(f)
    return None

def _save_progress(run_dir, last_idx, n_total):
    with open(os.path.join(run_dir, 'progress.json'), 'w') as f:
        json.dump({'last_completed_idx': last_idx, 'n_total': n_total,
                   'updated_at': time.strftime('%Y-%m-%dT%H:%M:%S')}, f)

def run_llm_on_dataset(runner, model_key, ds_name, test_samples, fewshot_pool, template, shots):
    run_id = _run_id(model_key, ds_name, template, shots)
    run_dir = os.path.join(PRED_DIR, run_id)
    os.makedirs(run_dir, exist_ok=True)

    progress = _load_progress(run_dir)
    start_idx = 0
    if progress and progress['last_completed_idx'] >= len(test_samples) - 1:
        print(f'  [{run_id}] already complete, skipping.')
        return
    if progress:
        start_idx = progress['last_completed_idx'] + 1
        print(f'  [{run_id}] resuming from index {start_idx}')

    few_shot_examples = sample_fewshot(fewshot_pool, shots, seed=cfg['seed'])
    remaining = test_samples[start_idx:]

    prompts = [build_prompt(template, s['text'], few_shot_examples) for s in remaining]

    pred_file = os.path.join(run_dir, 'predictions.jsonl')
    mode = 'a' if start_idx > 0 else 'w'

    with open(pred_file, mode) as f:
        batch_size = cfg['inference']['batch_size']
        for batch_start in range(0, len(prompts), batch_size):
            batch_prompts = prompts[batch_start : batch_start + batch_size]
            batch_samples = remaining[batch_start : batch_start + batch_size]
            results = runner.generate(batch_prompts, batch_size=batch_size,
                                      max_new_tokens=cfg['inference']['max_new_tokens'])
            for sample, (raw_out, latency_ms) in zip(batch_samples, results):
                label = parse(raw_out)
                pred = {'id': sample['id'], 'pred_label': label, 'raw_output': raw_out,
                        'parse_ok': label is not None, 'latency_ms': latency_ms}
                f.write(json.dumps(pred) + '\n')

            global_idx = start_idx + batch_start + len(batch_prompts) - 1
            if (global_idx + 1) % CHECKPOINT_EVERY == 0 or (batch_start + batch_size) >= len(prompts):
                f.flush()
                _save_progress(run_dir, global_idx, len(test_samples))

    print(f'  [{run_id}] done.')

In [ ]:
MODEL_KEYS = {
    'finllama': cfg['models']['finllama']['hf_id'],
    'llama31':  cfg['models']['llama31']['hf_id'],
}
TEMPLATES = cfg['prompts']['templates']
SHOTS     = cfg['prompts']['shots']

for model_key, hf_id in MODEL_KEYS.items():
    print(f'\n=== Loading {model_key} ({hf_id}) ===')
    runner = LLMRunner(hf_id=hf_id, load_in_4bit=cfg['models'][model_key]['load_in_4bit'],
                       seed=cfg['seed'])

    for ds_name, (test_samples, fewshot_pool) in datasets.items():
        for template in TEMPLATES:
            for shots in SHOTS:
                print(f'  {model_key} | {ds_name} | T={template} | shots={shots}')
                run_llm_on_dataset(runner, model_key, ds_name, test_samples,
                                   fewshot_pool, template, shots)

    runner.unload()
    print(f'=== {model_key} unloaded ===')

print('\nAll LLM runs complete.')